# Colab dots.ocr — the FAIR challenger test (full-page layout, CUDA)

dots.ocr (1.7B backbone, `rednote-hilab/dots.ocr`) emits document layout as JSON with
**tables as HTML** (carrying `colspan`/`rowspan`, which Surya 2 dropped). Apple-Silicon Stage A
(PROJECT_LOG §2.85) only completed ONE scored run — the hardest 124-DPI scan, in off-label
table-crop mode, on MPS (documented correctness risk). Its best-case document type, born-digital,
OOM'd before scoring. **This notebook is the fair test:** CUDA (numerically correct, unlike MPS),
full resolution, dots.ocr's DESIGNED full-page mode, on every document class.

Emits `predictions.json = {"<image.png>": "<raw model output>"}`, scored locally — identically to
the local engines:

```bash
uv run python scripts/score_dots_predictions.py --predictions predictions.json
```

**Steps:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run all cells → upload the page PNGs from
`eval/datasets/real/`, `eval/datasets/budget_textlayer/` and `eval/datasets/moc_gas/` (the same
files the local A/B scores) → the last cell downloads `predictions.json`.

Privacy: these are the financial PNGs; Colab (open model) is the path chosen over a commercial
cloud API — and every eval doc here is an already-published public bulletin.

> **T4 note:** Turing has no flash-attention-2. We use CUDA **sdpa**, which is correct AND
> memory-efficient — the point of Colab is escaping MPS, not chasing flash-attn. On a Pro L4/A100,
> vLLM (`vllm==0.9.1`) is the even-more-faithful config the model's own eval used.

In [ ]:
!pip install -q -U "transformers>=4.51,<4.58" accelerate torchvision pillow

In [ ]:
# Upload the page PNGs (ARDB born-digital, budget born-digital, moc_gas scan).
# Filenames are kept as-is and become the prediction keys, so they match the local GT.
from google.colab import files
uploaded = files.upload()
import glob
print(sorted(glob.glob('*.png')))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor, AutoConfig

REPO = "rednote-hilab/dots.ocr"

# Stage-A lesson: the VISION tower has its own attn setting, defaulting to
# flash_attention_2. On any non-flash GPU that must be set to sdpa explicitly, or
# it silently falls back to eager and materialises the full patch-attention matrix.
cfg = AutoConfig.from_pretrained(REPO, trust_remote_code=True)
cfg.vision_config.attn_implementation = "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    REPO, config=cfg, trust_remote_code=True,
    dtype=torch.bfloat16, attn_implementation="sdpa", low_cpu_mem_usage=True,
).to("cuda").eval()

# AutoProcessor is unusable under transformers 4.5x: DotsVLProcessor subclasses
# Qwen2_5_VLProcessor with a 3-arg __init__, but video_processor is now required.
# Build the two real pieces directly.
tok = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
improc = AutoImageProcessor.from_pretrained(REPO, trust_remote_code=True)
print("loaded", REPO)

In [ ]:
# dots.ocr's OFFICIAL full-page prompt (prompt_layout_all_en). Tables as HTML.
PROMPT = (
    "Please output the layout information from the PDF image, including each layout "
    "element's bbox, its category, and the corresponding text content within the bbox.\n\n"
    "1. Bbox format: [x1, y1, x2, y2]\n\n"
    "2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', "
    "'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', "
    "'Title'].\n\n"
    "3. Text Extraction & Formatting Rules:\n"
    "    - Picture: For the 'Picture' category, the text field should be omitted.\n"
    "    - Formula: Format its text as LaTeX.\n"
    "    - Table: Format its text as HTML.\n"
    "    - All Others (Text, Title, etc.): Format their text as Markdown.\n\n"
    "4. Constraints:\n"
    "    - The output text must be the original text from the image, with no translation.\n"
    "    - All layout elements must be sorted according to human reading order.\n\n"
    "5. Final Output: The entire output must be a single JSON object.\n"
)

In [ ]:
import json, glob, time
from PIL import Image

# Full resolution: T4 has 16GB, so we do NOT downscale the way MPS forced us to.
# max_new_tokens is generous because HTML for a dense table (e.g. 34x16 = 544 cells)
# is long; too small a cap truncates mid-table (a Stage-A failure mode).
MAX_NEW_TOKENS = 12000

def run_page(path):
    img = Image.open(path).convert("RGB")
    vis = improc(images=[img], return_tensors="pt")
    grid = vis["image_grid_thw"]
    # Qwen2-VL packing: image collapses to grid patches, then merge_size**2 patches
    # fuse into one token, so the <|imgpad|> placeholder repeats exactly that many times.
    n_img = int(grid.prod().item()) // (improc.merge_size ** 2)
    # dots.ocr's OWN chat format, NOT Qwen's <|im_start|> — the wrong one emits EOS instantly.
    prompt = ("<|user|><|img|>" + "<|imgpad|>" * n_img + "<|endofimg|>"
              + PROMPT + "<|endofuser|><|assistant|>")
    enc = tok([prompt], return_tensors="pt")
    inputs = {
        "input_ids": enc.input_ids.to("cuda"),
        "attention_mask": enc.attention_mask.to("cuda"),
        "pixel_values": vis["pixel_values"].to("cuda", dtype=torch.bfloat16),
        "image_grid_thw": grid.to("cuda"),
    }
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(trimmed, skip_special_tokens=True)
    n_new = int(trimmed.shape[0])
    flag = "  TRUNCATED(hit cap)" if n_new >= MAX_NEW_TOKENS else ""
    print(f"{path}: {img.size} -> {n_img} img tokens, {n_new} new tokens, "
          f"{time.perf_counter()-t0:.0f}s{flag}")
    return raw

preds = {}
for path in sorted(glob.glob("*.png")):
    preds[path] = run_page(path)

with open("predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)
print("wrote predictions.json")

In [ ]:
from google.colab import files
files.download("predictions.json")